======================================================================================================================

$$\text{This script plots a threshold-dynamics figure of the ODE model}$$
$$\text{Avulundiah Edwin Phiri}$$
$$\text{Department of Mathematics, Copperbelt University, Kitwe, Zambia}$$
$$\text{04/08/2026}$$

======================================================================================================================

In [5]:
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

DELTA = 1.0
GAMMA = 1.0
KAPPA = 1.0
N_EXPONENT = 5.0
X0 = 0.10
S0 = 0.50
T_END = 50.0


def rhs(_time, y, loss):
    s, x = y
    pore_fraction = 1.0 + X0 - x
    rate = s / (KAPPA + s)
    return [
        DELTA * pore_fraction**N_EXPONENT * (1.0 - s) - GAMMA * rate * x,
        (rate - loss) * x,
    ]


def pore_closure(_time, y, _loss):
    return 1.0 + X0 - y[1]


pore_closure.terminal = True
pore_closure.direction = -1


def main():
    times = np.linspace(0.0, T_END, 1001)
    scenarios = [
        (0.75, r"$\mathcal{R}_0=2/3$ (depletion)", "#bf2d56"),
        (0.25, r"$\mathcal{R}_0=2$ (persistence)", "#155b93"),
    ]
    fig, axes = plt.subplots(2, 1, figsize=(6.2, 5.0), sharex=True)
    for loss, label, color in scenarios:
        solution = solve_ivp(
            lambda t, y: rhs(t, y, loss),
            (0.0, T_END),
            (S0, X0),
            method="DOP853",
            rtol=1e-10,
            atol=1e-12,
            dense_output=True,
            events=lambda t, y: pore_closure(t, y, loss),
        )
        if not solution.success or solution.t[-1] < T_END - 1e-7:
            raise RuntimeError("Unexpected integration failure or pore closure")
        values = solution.sol(times)
        if np.min(1.0 + X0 - values[1]) <= 0:
            raise RuntimeError("Loss of admissibility in the plotted interval")
        for ax, row in zip(axes, values):
            ax.plot(times, row, lw=2.0, label=label, color=color)
        if loss == 0.75:
            expected_s, expected_x = 1.0, 0.0
        else:
            expected_s = loss * KAPPA / (1 - loss)
            H = DELTA * (1 - expected_s) / (GAMMA * loss)
            expected_x = brentq(lambda x: x-H*(1+X0-x)**N_EXPONENT, 0, 1+X0)
        error = max(abs(values[0, -1] - expected_s), abs(values[1, -1] - expected_x))
        print(f"lambda={loss:.2f}, R0={1/(loss*(1+KAPPA)):.3f}, "
              f"equilibrium=({expected_s:.6f},{expected_x:.6f}), "
              f"terminal error={error:.3e}")
    axes[0].set_ylabel("Substrate, $s(\\tau)$")
    axes[1].set_ylabel("Mineral loading, $x(\\tau)$")
    axes[1].set_xlabel("Dimensionless time, $\\tau$")
    axes[0].legend(loc="lower right", frameon=False, fontsize=9)
    for ax in axes:
        ax.grid(alpha=0.2)
        ax.set_xlim(0, T_END)
    axes[0].set_ylim(0, 1.05)
    axes[1].set_ylim(bottom=0)
    fig.tight_layout()
    fig.savefig(Path(__file__).resolve().parent / "threshold_dynamics_revised.pdf")
    fig.savefig(Path(__file__).resolve().parent / "threshold_dynamics_revised.png", dpi=300)


if __name__ == "__main__":
    main()


NameError: name '__file__' is not defined